# Phase 3 smoke test — custom markets × custom fleet

End-to-end check that the fleet model integrates correctly with **user-defined markets**.

Custom setup (in `data/`):

| File | Purpose |
|------|---------|
| `custom_markets.yaml` | 2 passenger markets + 1 freight (replacing the default 4) |
| `custom_fleet.yaml`   | Categories keyed to `regional` and `intercontinental` |
| `custom_aircraft_inventory.yaml` | Reference + new aircraft for both markets |

Market structure:

| Market id | Name | Type | RPK share 2019 | Energy share 2019 |
|-----------|------|------|---------------|------------------|
| `regional` | Regional | passenger | 62.3 % (SR+MR combined) | 52.19 % |
| `intercontinental` | Intercontinental | passenger | 37.7 % (= LR) | 32.81 % |
| `air_cargo` | Air Cargo | freight | — | 15 % |

**What we verify:**
1. Markets load and cross-validate against `custom_fleet.yaml` (no unknown/missing market IDs).
2. Fleet model initialises with the correct categories and subcategories for the 2 custom markets.
3. `fleet_model.compute()` runs without error and populates per-market energy columns.
4. Market traffic disciplines produce `regional_*` / `intercontinental_*` / `air_cargo_*` columns — not the legacy names.
5. Global aggregators (`rpk`, `ask`, `load_factor`) are still produced.

In [ ]:
from aeromaps import create_process

In [ ]:
process = create_process(configuration_file="data/config.yaml")

## 1 — Verify market registry

`process.markets` should hold exactly the 3 custom markets.  
Cross-validation against `custom_fleet.yaml` runs automatically at load time (unknown or missing passenger market IDs raise immediately).

In [ ]:
print("Markets loaded:")
for m in process.markets:
    print(f"  {m.id!r:25s}  name={m.name!r:22s}  type={m.traffic_type}")

assert set(process.markets.get_ids()) == {"regional", "intercontinental", "air_cargo"}, (
    "Unexpected market IDs"
)
assert len(process.markets.get(traffic_type="passenger")) == 2
assert len(process.markets.get(traffic_type="freight")) == 1
print("\n✓ Market registry OK")

## 2 — Verify fleet structure

`process.fleet_model.fleet.categories` must contain exactly one entry per passenger market,
keyed by the category display name, and each category must reference the correct `market_id`.

In [ ]:
categories = process.fleet_model.fleet.categories
print("Fleet categories:")
for cat_name, cat in categories.items():
    subcats = [sc.name for sc in cat.subcategories.values()]
    print(f"  [{cat.market_id}] {cat_name!r}  — subcategories: {subcats}")

assert set(c.market_id for c in categories.values()) == {"regional", "intercontinental"}, (
    "Category market_ids do not match custom markets"
)
assert len(categories) == 2, "Expected exactly 2 fleet categories"

# Check subcategory shares sum to 100 % per category
for cat_name, cat in categories.items():
    total_share = sum(float(sc.parameters.share or 0.0) for sc in cat.subcategories.values())
    assert abs(total_share - 100.0) < 1e-6, (
        f"Shares for {cat_name!r} sum to {total_share} (expected 100)"
    )

print("\n✓ Fleet structure OK")

## 3 — Run fleet model standalone

`fleet_model.compute()` runs before the GEMSEO MDA inside `process.compute()`.
Call it directly here so we can inspect its DataFrame without running the full MDA.

In [ ]:
process.fleet_model.fleet.all_aircraft_elements = (
    process.fleet_model.fleet.get_all_aircraft_elements()
)
process.fleet_model.compute()

fleet_df = process.fleet_model.df
print(f"Fleet DataFrame: {len(fleet_df)} rows × {len(fleet_df.columns)} columns")
print("\nSample columns:")
for col in sorted(fleet_df.columns)[:30]:
    print(f"  {col}")

In [ ]:
# Energy consumption and share columns — should use the display names "Regional" and "Intercontinental"
energy_cols = [c for c in fleet_df.columns if "energy_consumption" in c]
print("Energy-consumption columns in fleet_df:")
for c in energy_cols:
    print(f"  {c}")

assert any("Regional" in c for c in energy_cols), "No 'Regional' energy columns found"
assert any("Intercontinental" in c for c in energy_cols), (
    "No 'Intercontinental' energy columns found"
)

# Confirm legacy names are absent
legacy = ["Short Range", "Medium Range", "Long Range"]
bad = [c for c in fleet_df.columns if any(leg in c for leg in legacy)]
assert not bad, f"Legacy category names still present: {bad}"

print("\n✓ Fleet compute output uses custom market names")

In [ ]:
# Spot-check: mean energy consumption for each custom market in 2019 and 2050
years = [2019, 2030, 2050]
for cat_name in ["Regional", "Intercontinental"]:
    col = f"{cat_name}:energy_consumption"
    print(f"\n{col}:")
    print(fleet_df.loc[years, col])

## 4 — Full compute (market traffic disciplines via GEMSEO MDA)

`process.compute()` runs `fleet_model.compute()` then the full GEMSEO MDA.
The MDA includes the auto-created per-market RPK / ASK / load-factor / RTK disciplines
and their aggregators.

In [ ]:
import time

t0 = time.time()
process.compute()
print(f"compute() took {time.time() - t0:.2f} s")

## 5 — Inspect per-custom-market traffic outputs

In [ ]:
vector = process.data["vector_outputs"]

# Per-custom-market traffic columns
CUSTOM_MARKETS = ("regional_", "intercontinental_", "air_cargo_")
LEGACY_MARKETS = ("short_range_", "medium_range_", "long_range_", "freight_")

custom_traffic_cols = [
    c
    for c in vector.columns
    if c.startswith(CUSTOM_MARKETS)
    and any(tok in c for tok in ("rpk", "ask", "rtk", "load_factor"))
]
print("Per-custom-market traffic columns:")
for c in custom_traffic_cols:
    print(f"  {c}")

# Assert custom passenger-market names exist
assert any("regional_rpk" in c for c in vector.columns), "regional_rpk missing"
assert any("intercontinental_rpk" in c for c in vector.columns), "intercontinental_rpk missing"

# NOTE: RTKMarket still emits the legacy 'rtk' output name (Phase-2 known caveat;
# will be renamed to '<market_id>_rtk' in Phase 4).
assert "rtk" in vector.columns, "global rtk missing"
print("  rtk  (freight — still using legacy output name, Phase-4 rename pending)")

legacy_present = [c for c in vector.columns if c.startswith(LEGACY_MARKETS)]
assert not legacy_present, f"Legacy market columns present: {legacy_present}"

print("\n✓ Custom-market traffic columns present, no legacy names")

In [ ]:
# Global aggregators must still be present
assert "rpk" in vector.columns, "global rpk missing"
assert "ask" in vector.columns, "global ask missing"
assert "load_factor" in vector.columns, "global load_factor missing"

print("Global aggregator values at selected years:")
vector[["rpk", "ask", "load_factor"]].loc[[2019, 2030, 2050]]

In [ ]:
# Per-market traffic split at 2019 and 2050
traffic_cols = ["regional_rpk", "intercontinental_rpk"]
print("Per-market RPK (bn PKT):")
(vector[traffic_cols] / 1e9).loc[[2019, 2030, 2050]]

## 6 — Sanity-check: RPK shares vs YAML inputs

The per-market RPK in 2019 should match `rpk_share_2019` from `custom_markets.yaml`
(within floating-point tolerance).

- `regional`: 62.3 %
- `intercontinental`: 37.7 %

In [ ]:
total_rpk_2019 = vector["rpk"].loc[2019]
regional_share = vector["regional_rpk"].loc[2019] / total_rpk_2019 * 100
intercont_share = vector["intercontinental_rpk"].loc[2019] / total_rpk_2019 * 100

print(f"Regional share 2019:        {regional_share:.2f} %  (expected 62.3 %)")
print(f"Intercontinental share 2019: {intercont_share:.2f} %  (expected 37.7 %)")

assert abs(regional_share - 62.3) < 0.1, f"Regional RPK share mismatch: {regional_share:.3f}"
assert abs(intercont_share - 37.7) < 0.1, (
    f"Intercontinental RPK share mismatch: {intercont_share:.3f}"
)
print("\n✓ RPK shares match custom_markets.yaml inputs")